In [1]:
import os 
import requests
import psycopg2
from dotenv import load_dotenv
import logging
import pandas

load_dotenv()
log = logging.getLogger(__name__)

class Database:
    def __init__(self):
        self.db_host = os.getenv('DB_HOST')
        self.db_name = os.getenv('DB_NAME')
        self.db_user = os.getenv('DB_USER')
        self.db_password = os.getenv('DB_PASSWORD')
        self.db_sslmode = os.getenv('DB_SSLMODE')
        self.db_channelbinding = os.getenv('DB_CHANNELBIDING')
        self._compose_url()

    def _compose_url(self):
        self.database_url = (
            f"postgresql://{self.db_user}:{self.db_password}"
            f"@{self.db_host}/{self.db_name}"
            f"?sslmode={self.db_sslmode}&channel_binding={self.db_channelbinding}"
        )


    def execute(self, query:str, params: tuple = None) -> list:
        """ PERMET DE GERER UNE BDD LOCAL OU CLOUD (Actuellement uniquement cloud)"""
        #Execute HTTP NEON
        return self._execute_http(query, params)
        # Execute Psycop2 NEON
        # Execute local Database


    def _execute_http(self, query: str, params: tuple = None) -> list:
        """Connexion via HTTP API Neon (port 443)."""
        try:
            if params:
                query = query % tuple(
                    f"'{p}'" if isinstance(p, str) else str(p) for p in params
                )

            url = f"https://{self.db_host}/sql"
            headers = {
                "Content-Type": "application/json",
                "Neon-Connection-String": self.database_url
            }

            response = requests.post(url, json={"query": query}, headers=headers)
            data = response.json()

            # ── Vérification du status HTTP ──
            if response.status_code != 200:
                log.error(f"❌ HTTP {response.status_code} : {data.get('message', 'Erreur inconnue')}")
                raise Exception(f"HTTP {response.status_code} : {data.get('message')}")

            rows = data.get("rows", [])
            log.info(f"✅ Query OK ({len(rows)} lignes)")
            return rows

        except Exception as e:
            log.error(f"❌ HTTP Error : {e}")
            raise
        
    def bulk_insert(self, table: str, columns: list, data: list, batch_size: int = 3000) -> None:
        """Bulk insert avec découpage automatique en batches et échappement sécurisé."""
        
        total = len(data)
        inserted = 0

        # Découpe data en chunks de batch_size
        for i in range(0, total, batch_size):
            chunk = data[i:i + batch_size]

            values = []
            for row in chunk:
                row_formatted = []
                for v in row:
                    if v is None:
                        row_formatted.append("NULL")
                    elif isinstance(v, bool):
                        row_formatted.append("TRUE" if v else "FALSE")
                    elif isinstance(v, (int, float)):
                        row_formatted.append(str(v))
                    else:
                        # Échappe les apostrophes en les doublant pour PostgreSQL
                        escaped_str = str(v).replace("'", "''")
                        row_formatted.append(f"'{escaped_str}'")
                
                values.append(f"({', '.join(row_formatted)})")

            cols = ", ".join(f'"{c}"' for c in columns)
            all_values = ", ".join(values)
            query = f'INSERT INTO "{table}" ({cols}) VALUES {all_values};'

            # Appel à votre méthode qui exécute la requête HTTP Neon
            self.execute(query)
            
            inserted += len(chunk)
            log.info(f"✅ Batch {i // batch_size + 1} OK ({inserted}/{total} lignes)")

        log.info(f"✅ Bulk insert terminé ({total} lignes en {-(-total // batch_size)} requêtes)")





In [45]:
db = Database()

# ── Test 1 : SELECT initial ────────────────────────
print("\n--- TEST 1 : SELECT initial ---")
rows = db.execute("SELECT * FROM test;")
print(f"  {len(rows)} ligne(s) : {rows}")

# ── Test 2 : INSERT simple ─────────────────────────
print("\n--- TEST 2 : INSERT simple ---")
db.execute("INSERT INTO test (\"Nom\", timestamp) VALUES ('test_simple', '2026-05-29');")
rows = db.execute("SELECT * FROM test WHERE \"Nom\" = 'test_simple';")
print(f"  ✅ {len(rows)} ligne(s) insérée(s)")

# ── Test 3 : BULK INSERT 10 lignes ─────────────────
print("\n--- TEST 3 : BULK INSERT 10 lignes ---")
data_small = [(f"bulk{i}", "2026-05-29") for i in range(10)]
db.bulk_insert("test", ["Nom", "timestamp"], data_small)
rows = db.execute("SELECT COUNT(*) FROM test WHERE \"Nom\" LIKE 'bulk%';")
print(f"  ✅ {rows[0]['count']} ligne(s) bulk trouvée(s)")

# ── Test 4 : BULK INSERT 1500 lignes ──────────────
print("\n--- TEST 4 : BULK INSERT 1500 lignes ---")
data_large = [(f"bulk{i}", "2026-05-29") for i in range(1500)]
db.bulk_insert("test", ["Nom", "timestamp"], data_large)
rows = db.execute("SELECT COUNT(*) FROM test WHERE \"Nom\" LIKE 'bulk%';")
print(f"  ✅ {rows[0]['count']} ligne(s) bulk trouvée(s)")

# ── Test 5 : Types variés ──────────────────────────
print("\n--- TEST 5 : Types variés ---")
data_types = [
    ("string_test", "2026-05-29"),
    ("l'apostrophe", "2026-05-29"),  # test échappement
    ("null_test", None),              # test NULL
]
db.bulk_insert("test", ["Nom", "timestamp"], data_types)
rows = db.execute("SELECT * FROM test WHERE \"Nom\" IN ('string_test', 'l''apostrophe', 'null_test');")
print(f"  ✅ {len(rows)} ligne(s) avec types variés")
for row in rows:
    print(f"     {row}")

# ── Test 6 : Nettoyage ─────────────────────────────
print("\n--- TEST 6 : Nettoyage ---")
db.execute("DELETE FROM test WHERE \"Nom\" LIKE 'bulk%';")
db.execute("DELETE FROM test WHERE \"Nom\" IN ('test_simple', 'string_test', 'l''apostrophe', 'null_test');")
rows = db.execute("SELECT COUNT(*) FROM test;")
print(f"  ✅ Nettoyage OK — {rows[0]['count']} ligne(s) restante(s)")

2026-05-29 16:29:53,158 [DEBUG] Starting new HTTPS connection (1): ep-lingering-math-apltstip-pooler.c-7.us-east-1.aws.neon.tech:443



--- TEST 1 : SELECT initial ---


2026-05-29 16:29:53,437 [DEBUG] https://ep-lingering-math-apltstip-pooler.c-7.us-east-1.aws.neon.tech:443 "POST /sql HTTP/1.1" 200 258101
2026-05-29 16:29:53,504 [INFO] ✅ Query OK (4500 lignes)
2026-05-29 16:29:53,508 [DEBUG] Starting new HTTPS connection (1): ep-lingering-math-apltstip-pooler.c-7.us-east-1.aws.neon.tech:443


  4500 ligne(s) : [{'idtest': 3007, 'Nom': 'bulk0', 'timestamp': '2026-05-29'}, {'idtest': 3008, 'Nom': 'bulk1', 'timestamp': '2026-05-29'}, {'idtest': 3009, 'Nom': 'bulk2', 'timestamp': '2026-05-29'}, {'idtest': 3010, 'Nom': 'bulk3', 'timestamp': '2026-05-29'}, {'idtest': 3011, 'Nom': 'bulk4', 'timestamp': '2026-05-29'}, {'idtest': 3012, 'Nom': 'bulk5', 'timestamp': '2026-05-29'}, {'idtest': 3013, 'Nom': 'bulk6', 'timestamp': '2026-05-29'}, {'idtest': 3014, 'Nom': 'bulk7', 'timestamp': '2026-05-29'}, {'idtest': 3015, 'Nom': 'bulk8', 'timestamp': '2026-05-29'}, {'idtest': 3016, 'Nom': 'bulk9', 'timestamp': '2026-05-29'}, {'idtest': 3017, 'Nom': 'bulk10', 'timestamp': '2026-05-29'}, {'idtest': 3018, 'Nom': 'bulk11', 'timestamp': '2026-05-29'}, {'idtest': 3019, 'Nom': 'bulk12', 'timestamp': '2026-05-29'}, {'idtest': 3020, 'Nom': 'bulk13', 'timestamp': '2026-05-29'}, {'idtest': 3021, 'Nom': 'bulk14', 'timestamp': '2026-05-29'}, {'idtest': 3022, 'Nom': 'bulk15', 'timestamp': '2026-05-29'},

2026-05-29 16:29:53,750 [DEBUG] https://ep-lingering-math-apltstip-pooler.c-7.us-east-1.aws.neon.tech:443 "POST /sql HTTP/1.1" 200 74
2026-05-29 16:29:53,751 [INFO] ✅ Query OK (0 lignes)
2026-05-29 16:29:53,753 [DEBUG] Starting new HTTPS connection (1): ep-lingering-math-apltstip-pooler.c-7.us-east-1.aws.neon.tech:443
2026-05-29 16:29:53,997 [DEBUG] https://ep-lingering-math-apltstip-pooler.c-7.us-east-1.aws.neon.tech:443 "POST /sql HTTP/1.1" 200 489
2026-05-29 16:29:53,998 [INFO] ✅ Query OK (1 lignes)
2026-05-29 16:29:54,000 [DEBUG] Starting new HTTPS connection (1): ep-lingering-math-apltstip-pooler.c-7.us-east-1.aws.neon.tech:443


...

  ✅ 6010 ligne(s) bulk trouvée(s)

--- TEST 5 : Types variés ---


2026-05-29 16:29:55,344 [DEBUG] https://ep-lingering-math-apltstip-pooler.c-7.us-east-1.aws.neon.tech:443 "POST /sql HTTP/1.1" 200 74
2026-05-29 16:29:55,345 [INFO] ✅ Query OK (0 lignes)
2026-05-29 16:29:55,346 [INFO] ✅ Batch 1 OK (3/3 lignes)
2026-05-29 16:29:55,347 [INFO] ✅ Bulk insert terminé (3 lignes en 1 requêtes)
2026-05-29 16:29:55,348 [DEBUG] Starting new HTTPS connection (1): ep-lingering-math-apltstip-pooler.c-7.us-east-1.aws.neon.tech:443
2026-05-29 16:29:55,591 [DEBUG] https://ep-lingering-math-apltstip-pooler.c-7.us-east-1.aws.neon.tech:443 "POST /sql HTTP/1.1" 200 602
2026-05-29 16:29:55,592 [INFO] ✅ Query OK (3 lignes)
2026-05-29 16:29:55,594 [DEBUG] Starting new HTTPS connection (1): ep-lingering-math-apltstip-pooler.c-7.us-east-1.aws.neon.tech:443


  ✅ 3 ligne(s) avec types variés
     {'idtest': 9018, 'Nom': 'string_test', 'timestamp': '2026-05-29'}
     {'idtest': 9019, 'Nom': "l'apostrophe", 'timestamp': '2026-05-29'}
     {'idtest': 9020, 'Nom': 'null_test', 'timestamp': None}

--- TEST 6 : Nettoyage ---


2026-05-29 16:29:55,846 [DEBUG] https://ep-lingering-math-apltstip-pooler.c-7.us-east-1.aws.neon.tech:443 "POST /sql HTTP/1.1" 200 77
2026-05-29 16:29:55,847 [INFO] ✅ Query OK (0 lignes)
2026-05-29 16:29:55,849 [DEBUG] Starting new HTTPS connection (1): ep-lingering-math-apltstip-pooler.c-7.us-east-1.aws.neon.tech:443
2026-05-29 16:29:56,101 [DEBUG] https://ep-lingering-math-apltstip-pooler.c-7.us-east-1.aws.neon.tech:443 "POST /sql HTTP/1.1" 200 74
2026-05-29 16:29:56,102 [INFO] ✅ Query OK (0 lignes)
2026-05-29 16:29:56,104 [DEBUG] Starting new HTTPS connection (1): ep-lingering-math-apltstip-pooler.c-7.us-east-1.aws.neon.tech:443
2026-05-29 16:29:56,347 [DEBUG] https://ep-lingering-math-apltstip-pooler.c-7.us-east-1.aws.neon.tech:443 "POST /sql HTTP/1.1" 200 199
2026-05-29 16:29:56,349 [INFO] ✅ Query OK (1 lignes)


  ✅ Nettoyage OK — 0 ligne(s) restante(s)


In [ ]:
db = Database()
print("\n--- SELECTION --- ")
rows = db.execute("SELECT * FROM test;")
print(rows)

print("\n--- BulkINSERTION --- ")
data = [(f"bulk{i}", "2026-05-29") for i in range(15)]
db.bulk_insert('test', ['Nom', 'timestamp'], data)



print("\n--- SELECTION --- ")
rows = db.execute("SELECT * FROM test;")
print(rows)

2026-05-29 16:31:00,579 [DEBUG] Starting new HTTPS connection (1): ep-lingering-math-apltstip-pooler.c-7.us-east-1.aws.neon.tech:443



--- SELECTION --- 


2026-05-29 16:31:00,821 [DEBUG] https://ep-lingering-math-apltstip-pooler.c-7.us-east-1.aws.neon.tech:443 "POST /sql HTTP/1.1" 200 429
2026-05-29 16:31:00,821 [INFO] ✅ Query OK (0 lignes)
2026-05-29 16:31:00,824 [DEBUG] Starting new HTTPS connection (1): ep-lingering-math-apltstip-pooler.c-7.us-east-1.aws.neon.tech:443


[]

--- BulkINSERTION --- 


2026-05-29 16:31:01,089 [DEBUG] https://ep-lingering-math-apltstip-pooler.c-7.us-east-1.aws.neon.tech:443 "POST /sql HTTP/1.1" 200 75
2026-05-29 16:31:01,089 [INFO] ✅ Query OK (0 lignes)
2026-05-29 16:31:01,091 [INFO] ✅ Batch 1 OK (10/10 lignes)
2026-05-29 16:31:01,091 [INFO] ✅ Bulk insert terminé (10 lignes en 1 requêtes)
2026-05-29 16:31:01,093 [DEBUG] Starting new HTTPS connection (1): ep-lingering-math-apltstip-pooler.c-7.us-east-1.aws.neon.tech:443



--- SELECTION --- 


2026-05-29 16:31:01,346 [DEBUG] https://ep-lingering-math-apltstip-pooler.c-7.us-east-1.aws.neon.tech:443 "POST /sql HTTP/1.1" 200 979
2026-05-29 16:31:01,347 [INFO] ✅ Query OK (10 lignes)


[{'idtest': 9031, 'Nom': 'bulk0', 'timestamp': '2026-05-29'}, {'idtest': 9032, 'Nom': 'bulk1', 'timestamp': '2026-05-29'}, {'idtest': 9033, 'Nom': 'bulk2', 'timestamp': '2026-05-29'}, {'idtest': 9034, 'Nom': 'bulk3', 'timestamp': '2026-05-29'}, {'idtest': 9035, 'Nom': 'bulk4', 'timestamp': '2026-05-29'}, {'idtest': 9036, 'Nom': 'bulk5', 'timestamp': '2026-05-29'}, {'idtest': 9037, 'Nom': 'bulk6', 'timestamp': '2026-05-29'}, {'idtest': 9038, 'Nom': 'bulk7', 'timestamp': '2026-05-29'}, {'idtest': 9039, 'Nom': 'bulk8', 'timestamp': '2026-05-29'}, {'idtest': 9040, 'Nom': 'bulk9', 'timestamp': '2026-05-29'}]


In [52]:
import pandas as pd
rows = db.execute("SELECT * FROM test;")
df = pd.DataFrame(rows)
print(rows)

print(df)

2026-05-29 16:32:30,878 [DEBUG] Starting new HTTPS connection (1): ep-lingering-math-apltstip-pooler.c-7.us-east-1.aws.neon.tech:443
2026-05-29 16:32:31,122 [DEBUG] https://ep-lingering-math-apltstip-pooler.c-7.us-east-1.aws.neon.tech:443 "POST /sql HTTP/1.1" 200 979
2026-05-29 16:32:31,123 [INFO] ✅ Query OK (10 lignes)


[{'idtest': 9031, 'Nom': 'bulk0', 'timestamp': '2026-05-29'}, {'idtest': 9032, 'Nom': 'bulk1', 'timestamp': '2026-05-29'}, {'idtest': 9033, 'Nom': 'bulk2', 'timestamp': '2026-05-29'}, {'idtest': 9034, 'Nom': 'bulk3', 'timestamp': '2026-05-29'}, {'idtest': 9035, 'Nom': 'bulk4', 'timestamp': '2026-05-29'}, {'idtest': 9036, 'Nom': 'bulk5', 'timestamp': '2026-05-29'}, {'idtest': 9037, 'Nom': 'bulk6', 'timestamp': '2026-05-29'}, {'idtest': 9038, 'Nom': 'bulk7', 'timestamp': '2026-05-29'}, {'idtest': 9039, 'Nom': 'bulk8', 'timestamp': '2026-05-29'}, {'idtest': 9040, 'Nom': 'bulk9', 'timestamp': '2026-05-29'}]
   idtest    Nom   timestamp
0    9031  bulk0  2026-05-29
1    9032  bulk1  2026-05-29
2    9033  bulk2  2026-05-29
3    9034  bulk3  2026-05-29
4    9035  bulk4  2026-05-29
5    9036  bulk5  2026-05-29
6    9037  bulk6  2026-05-29
7    9038  bulk7  2026-05-29
8    9039  bulk8  2026-05-29
9    9040  bulk9  2026-05-29


In [48]:
rows = db.execute('TRUNCATE TABLE test')

2026-05-29 16:30:52,083 [DEBUG] Starting new HTTPS connection (1): ep-lingering-math-apltstip-pooler.c-7.us-east-1.aws.neon.tech:443
2026-05-29 16:30:52,336 [DEBUG] https://ep-lingering-math-apltstip-pooler.c-7.us-east-1.aws.neon.tech:443 "POST /sql HTTP/1.1" 200 79
2026-05-29 16:30:52,337 [INFO] ✅ Query OK (0 lignes)
